# Practical 05(a) — Ridge Regression

Study L2 regularization, coefficient shrinkage and model generalization.

## 1. Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.metrics import mean_squared_error, r2_score


## 2. Load multivariate dataset

In [ ]:
data = fetch_california_housing(as_frame=True)
df = data.frame.sample(n=6000, random_state=42)
X = df.drop(columns="MedHouseVal")
y = df["MedHouseVal"]


## 3. Split data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


## 4. Fit baseline and Ridge models

In [ ]:
baseline = make_pipeline(StandardScaler(), LinearRegression())
baseline.fit(X_train, y_train)
base_pred = baseline.predict(X_test)
rows = [{"Model": "Linear Regression", "Alpha": 0,
         "MSE": mean_squared_error(y_test, base_pred),
         "R2": r2_score(y_test, base_pred),
         "Coefficient L2": np.linalg.norm(baseline[-1].coef_)}]
models = {}
for alpha in [0.01, 0.1, 1, 10, 100]:
    model = make_pipeline(StandardScaler(), Ridge(alpha=alpha))
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    models[alpha] = model
    rows.append({"Model": "Ridge", "Alpha": alpha,
                 "MSE": mean_squared_error(y_test, pred),
                 "R2": r2_score(y_test, pred),
                 "Coefficient L2": np.linalg.norm(model[-1].coef_)})
pd.DataFrame(rows)


## 5. Compare coefficient shrinkage

In [ ]:
pd.DataFrame({
    "Feature": X.columns,
    "Linear Regression": baseline[-1].coef_,
    "Ridge alpha=10": models[10][-1].coef_
})


## 6. Interpretation

Ridge adds an L2 penalty. It shrinks coefficients toward zero without normally making them exactly zero, and can help reduce variance and sensitivity to correlated predictors.